# $B^+\to K^+\pi^+\pi^-$ benchmark with efficiency and background

The full paper-inspired amplitude model is retained; the high-level API only removes workflow boilerplate.


In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    BaBarFlatte, BackgroundSpec, DecayChannel, DecayModel, FitSession, LASS,
    NonResonant, Parameter, RealImag, Resonance, ToyBackground, enable_x64,
    generate_toy, plot_dalitz,
)
from dalitzplotfitter.background import FunctionalBackground
from dalitzplotfitter.efficiency import FunctionalEfficiency

enable_x64()


In [ ]:
channel = DecayChannel("B+", ("K+", "pi+", "pi-"))
truth_xy = {
    "Kstar892": (1.00, 0.00),
    "KpiS": (1.40, -0.60),
    "rho770": (0.65, 0.10),
    "f0_980": (-0.20, 1.00),
    "NR": (-0.50, 0.10),
}
truth = {}
def coefficient(name, fixed=False):
    x,y = truth_xy[name]
    if fixed: return RealImag(x,y)
    truth[f"{name}.x"], truth[f"{name}.y"] = x,y
    return RealImag(
        Parameter.coefficient(f"{name}.x",x,owner=name,step=0.01),
        Parameter.coefficient(f"{name}.y",y,owner=name,step=0.01),
    )
c={n:coefficient(n,fixed=(n=="Kstar892")) for n in truth_xy}
model=DecayModel(
    channel,[
        Resonance("Kstar892",(0,2),c["Kstar892"],mass=0.8958,width=0.0474,spin=1,resonance_radius=4.0,parent_radius=4.0),
        Resonance("KpiS",(0,2),c["KpiS"],lineshape=LASS(2.07,3.32,1.8),mass=1.425,width=0.270,spin=0,resonance_radius=4.0,parent_radius=4.0),
        Resonance("rho770",(1,2),c["rho770"],mass=0.7753,width=0.1491,spin=1,resonance_radius=4.0,parent_radius=4.0),
        Resonance("f0_980",(1,2),c["f0_980"],lineshape=BaBarFlatte(),mass=0.965,width=0.0,spin=0,resonance_radius=4.0,parent_radius=4.0),
        NonResonant(c["NR"]),
    ],normalization_method="square-dalitz",normalization_resolution=350,normalization_pair=(0,2),
)
efficiency=FunctionalEfficiency(lambda d:0.50+0.35*jnp.clip(d["s13"]/20.0,0,1))
background=FunctionalBackground(lambda d:0.35+0.8*jnp.clip(d["s23"]/25.0,0,1))
f_sig=Parameter("signal_fraction",0.76,bounds=(0.05,0.99),step=0.01)


In [ ]:
data=generate_toy(
    model,35_000,parameters=truth,efficiency=efficiency,signal_fraction=0.82,
    backgrounds=(ToyBackground("combinatorial",background),),
    seed=404,pool_size=280_000,
)
plot_dalitz(data,x="s13",y="s23",title="Selected B+ benchmark pseudo-data")
plt.show()


In [ ]:
session=FitSession(
    model,data,efficiency=efficiency,signal_fraction=f_sig,
    backgrounds=(BackgroundSpec("combinatorial",background),),
)
rng=np.random.default_rng(4041)
start={p.name:truth[p.name]+rng.normal(0,0.10) for p in model.parameters if not p.fixed}
start["signal_fraction"]=0.72
result=session.fit(start,simplex=True,ncall=60_000)
session.report(result,acceptance_weighted_fractions=True)
session.plot_projection(result,"s13")
plt.show()
session.plot_projection(result,"s23")
plt.show()
